# Primary Road Density Across ZCTAs: Single- vs Multi-County

**Research questions**
- **Primary:** How does primary road density differ across ZIP Code Tabulation Areas (ZCTAs), and does that pattern change depending on whether a ZCTA lies within one county or crosses county lines?
- **Secondary:** Which counties and ZCTAs have the highest and lowest primary road density (road km per km² of land)?




### Week 9: Data Preparation

In [ ]:
!wget -q https://www2.census.gov/geo/tiger/TIGER2024/ZCTA520/tl_2024_us_zcta520.zip
!unzip -q -o tl_2024_us_zcta520.zip -d zcta

import geopandas as gpd

zcta = gpd.read_file("zcta/tl_2024_us_zcta520.shp")
zcta.head()

,ZCTA5CE20,GEOID20,GEOIDFQ20,CLASSFP20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,geometry
0,47236,47236,860Z200US47236,B5,G6350,S,1029063,0,+39.1517426,-085.7252769,"POLYGON ((-85.7341 39.15597, -85.72794 39.1561..."
1,47870,47870,860Z200US47870,B5,G6350,S,8830,0,+39.3701518,-087.4735141,"POLYGON ((-87.47414 39.37016, -87.47409 39.370..."
2,47851,47851,860Z200US47851,B5,G6350,S,53326,0,+39.5735839,-087.2459559,"POLYGON ((-87.24769 39.5745, -87.24711 39.5744..."
3,47337,47337,860Z200US47337,B5,G6350,S,303089,0,+39.8027537,-085.4372850,"POLYGON ((-85.44357 39.80328, -85.44346 39.803..."
4,47435,47435,860Z200US47435,B5,G6350,S,13302,0,+39.2657557,-086.2951577,"POLYGON ((-86.29592 39.26547, -86.29592 39.266..."


In [ ]:
!wget -q https://www2.census.gov/geo/tiger/TIGER2024/PRIMARYROADS/tl_2024_us_primaryroads.zip
!unzip -q -o tl_2024_us_primaryroads.zip -d roads

import geopandas as gpd
roads = gpd.read_file("roads/tl_2024_us_primaryroads.shp")
roads.head()

,LINEARID,FULLNAME,RTTYP,MTFCC,geometry
0,1104430641961,State Hwy 42,S,S1100,"LINESTRING (-89.34466 31.35903, -89.34374 31.3..."
1,1104430639695,State Hwy 42,S,S1100,"LINESTRING (-89.32506 31.37999, -89.32826 31.3..."
2,1108296451323,State Hwy 67,S,S1100,"LINESTRING (-88.49978 43.14901, -88.49788 43.1..."
3,1108296451337,State Hwy 67,S,S1100,"LINESTRING (-88.49978 43.14878, -88.4982 43.14..."
4,1109097223374,State Hwy 114,S,S1100,"LINESTRING (-97.03217 32.92924, -97.03059 32.9..."


In [ ]:
!wget -q https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip
!unzip -q -o tl_2024_us_county.zip -d county

import geopandas as gpd
counties = gpd.read_file("county/tl_2024_us_county.shp")
print(counties.shape)
counties.head()

(3235, 19)


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,0500000US31039,Cuming,Cuming County,06,H1,G4020,None,None,None,A,1477563042,10772508,+41.9158651,-096.7885168,"POLYGON ((-96.55525 41.82892, -96.55524 41.827..."
1,53,069,01513275,53069,0500000US53069,Wahkiakum,Wahkiakum County,06,H1,G4020,None,None,None,A,680980773,61564428,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35,011,00933054,35011,0500000US35011,De Baca,De Baca County,06,H1,G4020,None,None,None,A,6016818941,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31,109,00835876,31109,0500000US31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,None,A,2169269508,22850511,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31,129,00835886,31129,0500000US31129,Nuckolls,Nuckolls County,06,H1,G4020,None,None,None,A,1489645201,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."


In [ ]:
import os
import pandas as pd

datasets = {
    "ZCTA":     (zcta,     "zcta/tl_2024_us_zcta520",       "tl_2024_us_zcta520.zip"),
    "Roads":    (roads,    "roads/tl_2024_us_primaryroads", "tl_2024_us_primaryroads.zip"),
    "Counties": (counties, "county/tl_2024_us_county",      "tl_2024_us_county.zip"),
}

def mb(path):
    return round(os.path.getsize(path) / 1e6, 1)

summary = pd.DataFrame({
    name: {
        "Rows":          f"{len(gdf):,}",
        "Shape type":    ", ".join(sorted(gdf.geom_type.unique())),
        "Columns":       len(gdf.columns) - 1,
        "CRS":           f"{gdf.crs.name} (EPSG:{gdf.crs.to_epsg()})",
        "Units":         gdf.crs.axis_info[0].unit_name,
        "Missing cells": f"{gdf.isna().sum().sum():,}",
        "Zip (MB)":      mb(zipf),
        ".shp (MB)":     mb(base + ".shp"),
        ".dbf (MB)":     mb(base + ".dbf"),
    }
    for name, (gdf, base, zipf) in datasets.items()
})
display(summary)

for name, (gdf, _, _) in datasets.items():
    gaps = gdf.isna().sum()
    gaps = gaps[gaps > 0]
    if len(gaps):
        print(f"\n{name}: missing values by column")
        print((gaps.astype(str) + " of " + f"{len(gdf):,}").to_string())

,ZCTA,Roads,Counties
Rows,"33,791","17,522","3,235"
Shape type,"MultiPolygon, Polygon",LineString,"MultiPolygon, Polygon"
Columns,10,4,18
CRS,NAD83 (EPSG:4269),NAD83 (EPSG:4269),NAD83 (EPSG:4269)
Units,degree,degree,degree
Missing cells,0,0,"6,310"
Zip (MB),528.8,38.3,83.9
.shp (MB),822.2,58.3,132.2
.dbf (MB),2.8,2.3,1.0



Counties: missing values by column
CSAFP       1894 of 3,235
CBSAFP      1320 of 3,235
METDIVFP    3096 of 3,235


## Week 10: Fixing Geometries and Defining Scope


In [ ]:
def fix_geometries(gdf, name):
    invalid = ~gdf.is_valid
    empty = gdf.geometry.is_empty | gdf.geometry.isna()
    print(f"{name:<9} invalid: {invalid.sum():>4}   empty: {empty.sum():>3}", end="")

    gdf = gdf[~empty].copy()
    gdf.loc[~gdf.is_valid, "geometry"] = gdf.loc[~gdf.is_valid].geometry.make_valid()

    print(f"   → still invalid after fix: {(~gdf.is_valid).sum()}")
    return gdf

zcta     = fix_geometries(zcta, "ZCTA")
roads    = fix_geometries(roads, "Roads")
counties = fix_geometries(counties, "Counties")

ZCTA      invalid:    0   empty:   0   → still invalid after fix: 0
Roads     invalid:    0   empty:   0   → still invalid after fix: 0
Counties  invalid:    0   empty:   0   → still invalid after fix: 0


In [ ]:
SCOPE = "50_states_dc"
TERRITORIES = ["60", "66", "69", "72", "78"]
AK_HI       = ["02", "15"]

drop = {"all": [], "50_states_dc": TERRITORIES, "contiguous": TERRITORIES + AK_HI}[SCOPE]

counties_f = counties[~counties["STATEFP"].isin(drop)].copy()

pts = gpd.GeoDataFrame(
    zcta[["ZCTA5CE20"]],
    geometry=gpd.points_from_xy(zcta["INTPTLON20"].astype(float), zcta["INTPTLAT20"].astype(float)),
    crs=zcta.crs,
)
zcta_state = gpd.sjoin(pts, counties[["STATEFP", "geometry"]], predicate="within")[["ZCTA5CE20", "STATEFP"]]
zcta_f = zcta.merge(zcta_state.drop_duplicates("ZCTA5CE20"), on="ZCTA5CE20", how="left")
zcta_f = zcta_f[zcta_f["STATEFP"].notna() & ~zcta_f["STATEFP"].isin(drop)].copy()

keep_area = counties_f[["geometry"]]
road_ids = gpd.sjoin(roads, keep_area, predicate="intersects")["LINEARID"].unique()
roads_f = roads[roads["LINEARID"].isin(road_ids)].copy()

print(f"Scope: {SCOPE}\n")
for name, before, after in [("ZCTA", zcta, zcta_f), ("Roads", roads, roads_f), ("Counties", counties, counties_f)]:
    print(f"{name:<9} {len(before):>7,} → {len(after):>7,}   (dropped {len(before) - len(after):,})")

Scope: 50_states_dc

ZCTA       33,791 →  33,642   (dropped 149)
Roads      17,522 →  17,273   (dropped 249)
Counties    3,235 →   3,144   (dropped 91)


## Week 11: Categorization


In [ ]:
z = zcta_f[["ZCTA5CE20", "geometry"]].to_crs(5070)
c = counties_f[["GEOID", "NAMELSAD", "STATEFP", "geometry"]].to_crs(5070)
z["zcta_area"] = z.area

pairs = gpd.sjoin(z, c, predicate="intersects")

county_geoms = c.geometry.loc[pairs["index_right"]]
pairs["overlap_area"] = pairs.geometry.intersection(county_geoms, align=False).area
pairs["share"] = pairs["overlap_area"] / pairs["zcta_area"]

MIN_SHARE = 0.01
real = pairs[pairs["share"] >= MIN_SHARE]

n_counties = real.groupby("ZCTA5CE20")["GEOID"].nunique().rename("n_counties")
zcta_f = zcta_f.drop(columns=["n_counties", "county_type"], errors="ignore").merge(
    n_counties, on="ZCTA5CE20", how="left"
)
zcta_f["county_type"] = zcta_f["n_counties"].apply(
    lambda n: "single-county" if n == 1 else "multi-county"
)

print(zcta_f["county_type"].value_counts().to_string())
print(f"\nMost counties touched by one ZCTA: {zcta_f['n_counties'].max()}")
print("\nDistribution of counties per ZCTA:")
print(zcta_f["n_counties"].value_counts().sort_index().to_string())

county_type
single-county    24240
multi-county      9402

Most counties touched by one ZCTA: 6

Distribution of counties per ZCTA:
n_counties
1    24240
2     7257
3     1853
4      276
5       14
6        2


## Week 12:

In [ ]:
import numpy as np
import pandas as pd

STATES = {
    "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE","11":"DC",
    "12":"FL","13":"GA","15":"HI","16":"ID","17":"IL","18":"IN","19":"IA","20":"KS","21":"KY",
    "22":"LA","23":"ME","24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO","30":"MT",
    "31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND","39":"OH",
    "40":"OK","41":"OR","42":"PA","44":"RI","45":"SC","46":"SD","47":"TN","48":"TX","49":"UT",
    "50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY",
}
REGIONS = {
    "Northeast": ["CT","ME","MA","NH","RI","VT","NJ","NY","PA"],
    "Midwest":   ["IL","IN","MI","OH","WI","IA","KS","MN","MO","NE","ND","SD"],
    "South":     ["DE","DC","FL","GA","MD","NC","SC","VA","WV","AL","KY","MS","TN","AR","LA","OK","TX"],
    "West":      ["AZ","CO","ID","MT","NV","NM","UT","WY","AK","CA","HI","OR","WA"],
}
state_to_region = {s: r for r, states in REGIONS.items() for s in states}

df = zcta_f.drop(columns="geometry").copy()
df["state"]     = df["STATEFP"].map(STATES).fillna("Territory")
df["region"]    = df["state"].map(state_to_region).fillna("Territory")
df["land_km2"]  = df["ALAND20"] / 1e6
df["log2_land"] = np.log2(df["land_km2"].clip(lower=0.01))
df["multi"]     = (df["county_type"] == "multi-county").astype(int)

In [ ]:
z = zcta_f[["ZCTA5CE20", "geometry"]].to_crs(5070)
r = roads_f[["LINEARID", "geometry"]].to_crs(5070)

# Cut roads at ZCTA edges and add up the km inside each ZCTA
hits = gpd.sjoin(r, z, predicate="intersects")
pieces = hits.geometry.intersection(z.geometry.loc[hits["index_right"]], align=False)
road_km = (pieces.length / 1000).groupby(hits["ZCTA5CE20"].values).sum()

df["road_km"] = df["ZCTA5CE20"].map(road_km).fillna(0)
df["road_density"] = df["road_km"] / df["land_km2"].replace(0, np.nan)

df = df[df["land_km2"] >= 1]   # skip tiny ZCTAs, where one highway makes density explode

In [ ]:
def road_summary(g):
    has_road = g[g["road_km"] > 0]
    return pd.Series({
        "ZCTAs":                    len(g),
        "% with a primary road":    round(100 * len(has_road) / len(g), 1),
        "Mean density (all)":       round(g["road_density"].mean(), 3),
        "Median density (if road)": round(has_road["road_density"].median(), 3),
    })

print("Overall")
display(df.groupby("county_type").apply(road_summary))

print("By region")
display(df.groupby(["region", "county_type"]).apply(road_summary))

Overall


/tmp/ipykernel_1469/70231001.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(df.groupby("county_type").apply(road_summary))


,ZCTAs,% with a primary road,Mean density (all),Median density (if road)
county_type,,,,
multi-county,9393.0,31.6,0.083,0.143
single-county,22629.0,36.9,0.263,0.384


By region


/tmp/ipykernel_1469/70231001.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(df.groupby(["region", "county_type"]).apply(road_summary))


ZCTAs  % with a primary road  Mean density (all)  \
region    county_type                                                        
Midwest   multi-county   4033.0                   26.2               0.056   
          single-county  5637.0                   34.3               0.214   
Northeast multi-county    775.0                   37.7               0.146   
          single-county  4882.0                   39.7               0.340   
South     multi-county   3736.0                   35.8               0.099   
          single-county  7339.0                   36.4               0.248   
West      multi-county    849.0                   33.5               0.084   
          single-county  4771.0                   38.0               0.265   

                         Median density (if road)  
region    county_type                              
Midwest   multi-county                      0.138  
          single-county                     0.321  
Northeast multi-county                      0.240  
          single-county                     0.496  
South     multi-county                      0.137  
          single-county                     0.352  
West      multi-county                      0.111  
          single-county                     0.364